# NKSK Climatic Water Deficit (CWD) Mapper
## Gary Ding &nbsp;&nbsp;&nbsp; May 14th, 2026

Computes annual **climatic water deficit (CWD = PET − AET)** for the NKSK region using
the Priestley-Taylor reference ET (PET) and actual ET (AET) monthly rasters from the
**Hawaii Evapotranspiration Atlas** (Giambelluca et al., 2014). Produces two maps:

1. CWD across NKSK (continuous raster)
2. Mean CWD along each pre-defined road segment in `road_segments.shp`

### How this notebook gets its data

A single **`NKSK_CWD_data.zip`** (~28 MB) bundles the four inputs the notebook needs
(PET grids, AET grids, NKSK boundary shapefile, and the road-segments shapefile).
Upload that zip to Drive once, then the cell below mounts Drive, copies the zip to
the local Colab VM, and unzips it. Outputs are written next to the unzipped data on
the Colab VM and (optionally) copied back to Drive at the end.

---


## Dependencies


In [ ]:
!pip install rasterio geopandas shapely -q
print("Ready.")

## Mount Drive and unzip the data bundle

Put `NKSK_CWD_data.zip` somewhere on Drive and point `ZIP_PATH` at it. The zip must
contain:
- `Priestley_ET0_mm_month_raster/` — monthly + annual PET Arc/Info grids
- `AET_mm_month_raster/` — monthly + annual AET Arc/Info grids
- `NKSK-boundaries/nksk.shp` (+ sidecar files) — NKSK study-area polygon
- `road_segments/road_segments.shp` (+ sidecar files) — pre-defined 1 km road segments

Two common Drive locations for the zip:
- `/content/drive/MyDrive/NKSK_CWD_data.zip` (personal My Drive root) — default
- `/content/drive/Shareddrives/Hawaii DST/GARY/Data/NKSK_CWD_data.zip` (shared drive)

Copying the zip to `/content/` and unzipping there is much faster than reading the
rasters directly from the mounted Drive on every cell.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, zipfile, time

# === Edit this to point at where you uploaded NKSK_CWD_data.zip ===
ZIP_PATH = "/content/drive/MyDrive/NKSK_CWD_data.zip"

# Local working directory on the Colab VM (fast disk)
WORK_DIR  = "/content/NKSK_CWD_work"
UNZIP_DIR = os.path.join(WORK_DIR, "data")   # raw extraction target
OUT_DIR   = os.path.join(WORK_DIR, "CWD_outputs")
os.makedirs(UNZIP_DIR, exist_ok=True)
os.makedirs(OUT_DIR,   exist_ok=True)

assert os.path.exists(ZIP_PATH), (
    f"Couldn't find {ZIP_PATH}. Upload NKSK_CWD_data.zip to Drive and "
    f"update ZIP_PATH to match."
)

# ── 1. Copy zip to the VM ────────────────────────────────────────────────────
local_zip = "/content/NKSK_CWD_data.zip"
if not os.path.exists(local_zip) or os.path.getsize(local_zip) != os.path.getsize(ZIP_PATH):
    t0 = time.time()
    shutil.copy(ZIP_PATH, local_zip)
    print(f"Copied zip in {time.time()-t0:.1f}s ({os.path.getsize(local_zip)/1e6:.1f} MB)")
else:
    print(f"Zip already at {local_zip}")

# ── 2. Unzip — re-extract whenever the zip changes ───────────────────────────
sentinel     = os.path.join(UNZIP_DIR, ".unzipped")
current_size = os.path.getsize(local_zip)

sentinel_size = None
if os.path.exists(sentinel):
    try:
        sentinel_size = int(open(sentinel).read().strip())
    except ValueError:
        sentinel_size = None  # old 'ok' sentinel — treat as stale

if sentinel_size == current_size:
    print(f"Zip unchanged ({current_size/1e6:.1f} MB) — skipping extraction.")
else:
    if sentinel_size is not None:
        print(f"Zip changed ({sentinel_size/1e6:.1f} MB → {current_size/1e6:.1f} MB). "
              "Clearing and re-extracting...")
    else:
        print("Extracting zip...")
    shutil.rmtree(UNZIP_DIR)
    os.makedirs(UNZIP_DIR)
    t0 = time.time()
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(UNZIP_DIR)
    open(sentinel, "w").write(str(current_size))
    print(f"Extracted in {time.time()-t0:.1f}s")

# ── 3. Resolve DATA_DIR — handle zips that extract into a single subfolder ───
# Macs often wrap everything in a top-level folder; __MACOSX is metadata noise.
real_entries = [
    e for e in os.listdir(UNZIP_DIR)
    if not e.startswith(".") and e != "__MACOSX"
]
if len(real_entries) == 1 and os.path.isdir(os.path.join(UNZIP_DIR, real_entries[0])):
    DATA_DIR = os.path.join(UNZIP_DIR, real_entries[0])
    print(f"Zip contains a top-level folder — using DATA_DIR = {DATA_DIR}")
else:
    DATA_DIR = UNZIP_DIR

print("\nContents of DATA_DIR:")
for name in sorted(os.listdir(DATA_DIR)):
    if not name.startswith(".") and name != "__MACOSX":
        print("  ", name)


## Configuration

Resolved input paths. Everything below derives from `DATA_DIR`.


In [ ]:
PET_DIR          = os.path.join(DATA_DIR, "Priestley_ET0_mm_month_raster")
AET_DIR          = os.path.join(DATA_DIR, "AET_mm_month_raster")
NKSK_SHP         = os.path.join(DATA_DIR, "NKSK-boundaries", "nksk.shp")
ROAD_SEGMENTS_SHP = os.path.join(DATA_DIR, "road_segments", "road_segments.shp")

# Sampling interval for CWD assignment along each road segment
SAMPLE_SPACING_M = 50  # metres

missing = [p for p in [PET_DIR, AET_DIR, NKSK_SHP, ROAD_SEGMENTS_SHP] if not os.path.exists(p)]
assert not missing, "Missing inputs:\n  " + "\n  ".join(missing)
print("Config OK. Outputs will go to:", OUT_DIR)


## Load PET and AET rasters

Each Arc/Info Grid folder can be opened directly with `rasterio.open(<folder>)` — rasterio
detects the `hdr.adf` and reads the raw `w001001.adf` block.


In [ ]:
import numpy as np, rasterio

MONTHS = ["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"]

def open_grid(parent, prefix, period):
    """period is 'ann' or one of MONTHS; folder name = f'{prefix}_{period}'."""
    return rasterio.open(os.path.join(parent, f"{prefix}_{period}"))

# Sanity-check a single month
with open_grid(PET_DIR, "pr0_mm", "jan") as pet_jan, open_grid(AET_DIR, "aet_mm", "jan") as aet_jan:
    print("PET jan:", pet_jan.crs, pet_jan.shape, "res", pet_jan.res, "mm/month range",
          float(pet_jan.read(1, masked=True).min()), "→",
          float(pet_jan.read(1, masked=True).max()))
    print("AET jan:", aet_jan.crs, aet_jan.shape, "res", aet_jan.res, "mm/month range",
          float(aet_jan.read(1, masked=True).min()), "→",
          float(aet_jan.read(1, masked=True).max()))

## Clip PET and AET to NKSK, compute CWD

CWD = **PET − AET**. We compute it two ways for cross-checking:

- **From the annual rasters** (`pr0_mm_ann − aet_mm_ann`) — fast.
- **Summing monthly differences** (`Σ_m (PET_m − AET_m)`) — confirms the annual layers
  match what we'd get by summing monthly.

Both rasters are on the same 0.00225° (~250 m) statewide grid in EPSG:4326, so no
resampling is needed before subtraction.


In [ ]:
import geopandas as gpd
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping

nksk = gpd.read_file(NKSK_SHP)
print("NKSK CRS:", nksk.crs, "polygons:", len(nksk))

def clip_to_nksk(grid_ds, nksk_gdf):
    """Clip a rasterio dataset to NKSK polygon. Returns (array, transform, nodata)."""
    geoms = list(nksk_gdf.to_crs(grid_ds.crs).geometry.map(mapping))
    arr, tx = rio_mask(grid_ds, geoms, crop=True)
    return arr[0], tx, grid_ds.nodata

# --- Approach A: annual rasters ---
with open_grid(PET_DIR, "pr0_mm", "ann") as pet_ann, open_grid(AET_DIR, "aet_mm", "ann") as aet_ann:
    pet_a, tx, pet_nd = clip_to_nksk(pet_ann, nksk)
    aet_a, _,  aet_nd = clip_to_nksk(aet_ann, nksk)
    raster_crs = pet_ann.crs

invalid = (pet_a == pet_nd) | (aet_a == aet_nd) | np.isnan(pet_a) | np.isnan(aet_a)
cwd_ann = np.where(invalid, np.nan, pet_a - aet_a)

print(f"\nCWD (from annual rasters): {np.nanmin(cwd_ann):.0f} – {np.nanmax(cwd_ann):.0f} mm/yr, "
      f"mean {np.nanmean(cwd_ann):.0f}, valid pixels {(~invalid).sum():,}")

# --- Approach B: sum monthly differences ---
cwd_monthly_sum = np.zeros_like(cwd_ann)
for m in MONTHS:
    with open_grid(PET_DIR, "pr0_mm", m) as pet_m, open_grid(AET_DIR, "aet_mm", m) as aet_m:
        p_m, _, _ = clip_to_nksk(pet_m, nksk)
        a_m, _, _ = clip_to_nksk(aet_m, nksk)
    inv_m = (p_m == pet_nd) | (a_m == aet_nd) | np.isnan(p_m) | np.isnan(a_m)
    cwd_monthly_sum = cwd_monthly_sum + np.where(inv_m, 0.0, p_m - a_m)
cwd_monthly_sum = np.where(invalid, np.nan, cwd_monthly_sum)

print(f"CWD (Σ monthly):           {np.nanmin(cwd_monthly_sum):.0f} – {np.nanmax(cwd_monthly_sum):.0f} mm/yr, "
      f"mean {np.nanmean(cwd_monthly_sum):.0f}")
print(f"Mean abs difference (A vs B): {np.nanmean(np.abs(cwd_ann - cwd_monthly_sum)):.3f} mm")

## Save the CWD raster


In [ ]:
out_tif = os.path.join(OUT_DIR, "NKSK_CWD_annual_mm.tif")
with rasterio.open(
    out_tif, "w",
    driver="GTiff",
    height=cwd_ann.shape[0], width=cwd_ann.shape[1],
    count=1, dtype="float32",
    crs=raster_crs, transform=tx,
    nodata=np.float32(-9999),
    compress="deflate",
) as dst:
    out = np.where(np.isnan(cwd_ann), -9999, cwd_ann).astype("float32")
    dst.write(out, 1)
print("Wrote", out_tif)

## Plot 1 — CWD across NKSK


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import rasterio.plot

cwd_cmap = LinearSegmentedColormap.from_list(
    "cwd", ["#2c7bb6", "#abd9e9", "#ffffbf", "#fdae61", "#d7191c"], N=256
)

fig, ax = plt.subplots(figsize=(9, 9))
vmin, vmax = np.nanpercentile(cwd_ann, [2, 98])
im = ax.imshow(
    cwd_ann,
    extent=rasterio.plot.plotting_extent(cwd_ann, tx),
    cmap=cwd_cmap, vmin=vmin, vmax=vmax,
)
nksk.to_crs(raster_crs).boundary.plot(ax=ax, edgecolor="black", linewidth=1.2)

cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
cbar.set_label("Annual CWD (mm)")

ax.set_title("Climatic Water Deficit — NKSK\nPET − AET, annual (mm)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "NKSK_CWD_map.png"), dpi=200, bbox_inches="tight")
plt.show()
print("Saved", os.path.join(OUT_DIR, "NKSK_CWD_map.png"))

## Sample CWD along road segments

Each segment in `road_segments.shp` already represents a discrete ~1 km section of a
named road inside NKSK. CWD is assigned by:

1. Reprojecting each segment to the raster CRS (EPSG:4326).
2. Interpolating sample points every **50 m** along the segment (using the native UTM
   Zone 5N / EPSG:32605 projection for accurate spacing).
3. Sampling the annual PET and AET rasters at each point and computing CWD = PET − AET.
4. Taking the **mean CWD** across all valid sample points as the segment's CWD value.


In [ ]:
segments = gpd.read_file(ROAD_SEGMENTS_SHP)  # EPSG:32605 (UTM Zone 5N, metres)
print(f"Road segments loaded: {len(segments)} rows")
print(f"Columns: {list(segments.columns)}")
print(f"CRS: {segments.crs}")
print()
print(segments.groupby('road_name')['seg_id'].count().rename('n_segments')
           .sort_values(ascending=False).to_string())


In [ ]:
from shapely.geometry import LineString, MultiLineString

def explode_lines(geom):
    """Return a flat list of LineString parts from a LineString or MultiLineString."""
    if isinstance(geom, LineString):
        return [geom]
    if isinstance(geom, MultiLineString):
        return list(geom.geoms)
    return []

def points_along_metric(line, spacing_m):
    """Interpolate points every spacing_m metres along a projected (metric) LineString."""
    n = max(2, int(line.length // spacing_m) + 1)
    return [line.interpolate(d) for d in np.linspace(0, line.length, n)]

# segments.crs is already EPSG:32605 (metric) — no unit conversion needed.
# For raster sampling we also need a version in the raster CRS (EPSG:4326).
segments_geo = segments.to_crs(raster_crs)

mean_cwd_per_segment = []

with rasterio.open(os.path.join(PET_DIR, "pr0_mm_ann")) as pet_ds, \
     rasterio.open(os.path.join(AET_DIR, "aet_mm_ann")) as aet_ds:

    for geom_utm, geom_geo in zip(segments.geometry, segments_geo.geometry):
        # Interpolate sample points in metric CRS for accurate spacing
        pts_utm = []
        for line in explode_lines(geom_utm):
            pts_utm += points_along_metric(line, SAMPLE_SPACING_M)

        if not pts_utm:
            mean_cwd_per_segment.append(np.nan)
            continue

        # Reproject sample points to raster CRS for pixel lookup
        pts_raster = gpd.GeoSeries(pts_utm, crs=segments.crs).to_crs(raster_crs)
        coords = [(p.x, p.y) for p in pts_raster]

        pet_vals = np.array([v[0] for v in pet_ds.sample(coords)], dtype="float64")
        aet_vals = np.array([v[0] for v in aet_ds.sample(coords)], dtype="float64")

        bad = ((pet_vals == pet_ds.nodata) | (aet_vals == aet_ds.nodata) |
               np.isnan(pet_vals) | np.isnan(aet_vals))
        cwd_vals = np.where(bad, np.nan, pet_vals - aet_vals)

        mean_cwd_per_segment.append(
            float(np.nanmean(cwd_vals)) if np.any(~bad) else np.nan
        )

segments["cwd_mm"] = mean_cwd_per_segment

print(f"Segments with valid CWD: {segments['cwd_mm'].notna().sum()} / {len(segments)}")
print()
print(segments.groupby('road_name')['cwd_mm']
              .agg(['mean','min','max'])
              .round(0)
              .sort_values('mean', ascending=False)
              .to_string())


## Plot 2 — CWD along road segments


In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

# Faint CWD raster backdrop
ax.imshow(
    cwd_ann,
    extent=rasterio.plot.plotting_extent(cwd_ann, tx),
    cmap=cwd_cmap, vmin=vmin, vmax=vmax, alpha=0.35,
)
nksk.to_crs(raster_crs).boundary.plot(ax=ax, edgecolor="black", linewidth=1.0)

# Road segments coloured by mean CWD
segments_geo = segments.to_crs(raster_crs)
v_lo, v_hi = np.nanpercentile(segments["cwd_mm"].dropna(), [2, 98])
segments_geo.plot(
    ax=ax, column="cwd_mm", cmap=cwd_cmap,
    vmin=v_lo, vmax=v_hi, linewidth=2.5, legend=False,
)

import matplotlib.cm as cm, matplotlib.colors as mcolors
sm = cm.ScalarMappable(norm=mcolors.Normalize(vmin=v_lo, vmax=v_hi), cmap=cwd_cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.04, pad=0.02)
cbar.set_label("Mean annual CWD along segment (mm)")

ax.set_title("CWD along NKSK road segments\n"
             f"(mean of {SAMPLE_SPACING_M} m sample points, PET − AET annual)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "NKSK_CWD_roads.png"), dpi=200, bbox_inches="tight")
plt.show()
print("Saved", os.path.join(OUT_DIR, "NKSK_CWD_roads.png"))


## Save per-segment table and copy outputs back to Drive

The CSV ranks every segment by mean CWD; the GeoJSON preserves geometries for
QGIS / web-map reuse. The final cell mirrors `CWD_outputs/` back to Drive next to
the input zip so results survive when the Colab VM is recycled.


In [ ]:
out_csv = os.path.join(OUT_DIR, "NKSK_road_segments_CWD.csv")
segments[["seg_id", "road_name", "length_m", "cwd_mm"]].to_csv(out_csv, index=False)
print("Wrote", out_csv)

out_geojson = os.path.join(OUT_DIR, "NKSK_road_segments_CWD.geojson")
segments.to_crs(4326).to_file(out_geojson, driver="GeoJSON")
print("Wrote", out_geojson)


In [ ]:
# Copy CWD_outputs/ back to Drive, next to the input zip
drive_out = os.path.join(os.path.dirname(ZIP_PATH), "CWD_outputs")
os.makedirs(drive_out, exist_ok=True)
for f in os.listdir(OUT_DIR):
    src = os.path.join(OUT_DIR, f)
    dst = os.path.join(drive_out, f)
    shutil.copy(src, dst)
print("Mirrored outputs to", drive_out)

## Summary

| Output | Location on Drive (after final cell) |
|---|---|
| CWD raster | `<zip parent>/CWD_outputs/NKSK_CWD_annual_mm.tif` |
| CWD map figure | `<zip parent>/CWD_outputs/NKSK_CWD_map.png` |
| CWD-along-segments figure | `<zip parent>/CWD_outputs/NKSK_CWD_roads.png` |
| Per-segment CWD table | `<zip parent>/CWD_outputs/NKSK_road_segments_CWD.csv` |
| Per-segment CWD geometry | `<zip parent>/CWD_outputs/NKSK_road_segments_CWD.geojson` |

### Notes & assumptions

- **PET source**: Priestley-Taylor reference ET (`pr0_mm_*`), monthly mm. Statewide ~250 m grid.
- **AET source**: actual ET (`aet_mm_*`), same grid, Hawaii Evapotranspiration Atlas (Giambelluca et al., 2014).
- **CWD definition**: PET − AET on the annual rasters; the monthly-sum cross-check confirmed the annual layers match Σ monthly to <1 mm/pixel.
- **Road segments**: `road_segments.shp` — 280 pre-defined ~1 km segments across 10 named roads in NKSK, in EPSG:32605 (UTM Zone 5N, metres).
- **CWD assignment**: mean of nearest-pixel CWD values sampled at 50 m intervals along each segment. Edit `SAMPLE_SPACING_M` in the Config cell to change resolution.
